In [ ]:
# Tropical Cyclone Intensity Decay & Validation
# Author: Siméon Vareilles
# Description: Validates the stochastic model by comparing simulated return periods against historical observations.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import pickle
import math
import statistics as stat
from tqdm import tqdm
from scipy.io import loadmat
from os.path import exists

# Config 

TC_DATA_DIR = os.getenv("TC_DATA_DIR", "../data") # Note: points to parent data folder if inside 'analysis'
MPI_DIR = os.path.join(TC_DATA_DIR, "mpi_out")

print(f"Loading data from: {TC_DATA_DIR}")

# Helper to load data safely
def get_data_path(filename):
    path = os.path.join(TC_DATA_DIR, filename)
    if not os.path.exists(path):
        print(f"Warning: {filename} not found at {path}")
    return path

In [30]:
plt.rcParams.update({
"text.usetex": True,
'font.size' : 20,
"font.family": "lmodern",
"ps.distiller.res": 100000})
plt.rcParams['figure.dpi'] = 250
plt.rcParams['savefig.dpi'] = 250
plt.style.use('seaborn-pastel')
plt.rcParams["figure.figsize"] = [7, 4]
plt.rcParams['axes.linewidth'] = 1.5 
plt.rcParams['xtick.labelsize'] = 18
plt.rcParams['ytick.labelsize'] = 18

In [ ]:
def calculate_return_period(data_values, years_of_record):
    """
    Calculates Return Period: T = (Years + 1) / Rank
    """
    # Sort descending (Strongest first)
    sorted_data = np.sort(data_values)[::-1]
    n = len(sorted_data)
    rank = np.arange(1, n + 1)
    
    # Return Period Calculation
    return_period = (years_of_record + 1) / rank
    return return_period, sorted_data

In [ ]:
# Main Analysis

print(f"Loading data from {TC_DATA_DIR}...")

try:
    # 1. Load OBSERVED Data
    with open(get_data_path("LMItracks_landfall.pkl"), "rb") as f:
        obs_data = pickle.load(f)
    
    # 2. Load SIMULATED Data
    sim_file = get_data_path("Simulated_Tracks.pkl") 
    
    if os.path.exists(sim_file):
        with open(sim_file, "rb") as f:
            sim_data = pickle.load(f)

        # DATA ACCESS SAFEGUARD 
        # 1. Get Observed Vmax 
        if hasattr(obs_data, 'vmax'):
            obs_vmax = np.array(obs_data.vmax)
        else:
            # Fallback for dictionaries
            obs_vmax = np.array(obs_data['vmax'])

        # Get Simulated Vmax
        
        if hasattr(sim_data, 'vmax'):
            sim_vmax = np.array(sim_data.vmax)
        elif isinstance(sim_data, dict) and 'vmax' in sim_data:
            sim_vmax = np.array(sim_data['vmax'])
        else:
            print("Error: Could not find 'vmax' in simulated data structure.")
            print(f"Available keys/attributes: {dir(sim_data)}")
            raise ValueError("Missing Vmax data")

        print(f"Loaded {len(obs_vmax)} observed storms.")
        print(f"Loaded {len(sim_vmax)} simulated storms.")

        # --- CALCULATE CURVES ---
        
        YEARS_OBS = 40
        YEARS_SIM = 1000 #Adjust years simulation length

        obs_rp, obs_v = calculate_return_period(obs_vmax, years_of_record=YEARS_OBS)
        sim_rp, sim_v = calculate_return_period(sim_vmax, years_of_record=YEARS_SIM)

        # --- PLOTTING ---
        plt.figure(figsize=(10, 7))
        
        # Plotting Linear (plt.plot) to match original code EXACTLY.
        # Note: If you want scientific standard, change to plt.semilogx()
        plt.plot(sim_rp, sim_v, '-', color='royalblue', alpha=0.8, label=f'Stochastic Model ({YEARS_SIM} yrs)')
        plt.plot(obs_rp, obs_v, 'o', color='firebrick', markersize=6, label=f'Historical ({YEARS_OBS} yrs)')
        
        plt.xlabel("Return Period (Years)")
        plt.ylabel("Maximum Wind Speed ($V_{max}$) [m/s]")
        plt.title("Model Validation: Return Period Comparison")
        plt.legend()
        plt.grid(True, which="both", alpha=0.3)
        
        plt.savefig(os.path.join(TC_DATA_DIR, "Validation_ReturnPeriod.png"))
        print("Plot saved successfully.")
        plt.show()

    else:
        print(f"File not found: {sim_file}")

except Exception as e:
    print(f"An error occurred: {e}")